# StateGraph: Nodes, Edges, and State

**What you will build:** a graph by hand — the real LangGraph primitive. We'll implement the **routing** pattern from 0.4 as a graph, so you can compare "a few `if`s in Python" with "an explicit state machine" and feel when the graph is worth it.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/21_stategraph_fundamentals.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## The three pieces

- **State** — a `TypedDict` that every node reads from and writes to. It's the shared memory of the graph.
- **Nodes** — plain functions `state -> partial state update`.
- **Edges** — how control flows between nodes; **conditional edges** branch based on the state.

We'll build: classify a support ticket, then route to the matching specialist node.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    text: str        # the incoming ticket
    category: str    # filled by the classifier
    reply: str       # filled by a specialist

def classify(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content": "Classify as ONE word: billing, technical, or other."},
                    {"role": "user", "content": state["text"]}])
    return {"category": r.content.strip().lower()}

def billing(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content": "You are a billing specialist. Be precise."},
                    {"role": "user", "content": state["text"]}])
    return {"reply": r.content}

def technical(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content": "You are a technical support engineer. Give steps."},
                    {"role": "user", "content": state["text"]}])
    return {"reply": r.content}

def other(state: State) -> dict:
    return {"reply": "Forwarding your message to a human agent."}

## Wire the nodes with edges

`add_conditional_edges` is the branch: a router function returns the name of the next node.

In [ ]:
def route(state: State) -> str:
    c = state["category"]
    if "billing" in c:   return "billing"
    if "technical" in c: return "technical"
    return "other"

builder = StateGraph(State)
builder.add_node("classify", classify)
builder.add_node("billing", billing)
builder.add_node("technical", technical)
builder.add_node("other", other)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route,
                              {"billing": "billing", "technical": "technical", "other": "other"})
for n in ("billing", "technical", "other"):
    builder.add_edge(n, END)

graph = builder.compile()

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))     # PNG via mermaid.ink (needs internet)
except Exception:
    print("Mermaid image unavailable, showing text instead:\n")
    print(graph.get_graph().draw_mermaid())                  # pure text (Mermaid DSL), no extra deps

Run it — the state flows in, gets a `category`, branches, and comes out with a `reply`:

In [ ]:
out = graph.invoke({"text": "I was charged twice for my subscription this month."})
print("category:", out["category"])
print("reply:", out["reply"][:200], "...")

You built the same routing logic as 0.4 — but now it's an explicit, drawable object, and the `State` carries data cleanly between steps. For three branches that's arguably heavier than plain `if`s. The graph pays off when the flow has loops, parallel paths, human pauses, or many nodes — everything coming in 2.3–2.6.

::::{dropdown} 🔧 Under the hood: state updates and the "merge problem"
:color: info

Each node returns a **partial** update (`{"category": ...}`) that LangGraph merges into the state. By default a key is *overwritten*. If two nodes write the same key (e.g. in parallel branches), the last write wins and you silently lose data. To *accumulate* instead, annotate the field with a reducer:

```python
from typing import Annotated
import operator
class State(TypedDict):
    findings: Annotated[list, operator.add]   # appends instead of overwriting
```

This is the same idea behind `add_messages`, the reducer that makes a `messages` list grow correctly — which we use in the next notebook.
::::

**What's next:** in **2.2** we build the **agent loop itself as a graph** — the ReAct cycle from 0.3, drawn with a real cycle in it.